# Geldium Delinquency Analysis: Data Checks and Baseline Model
*Tata GenAI Powered Data Analytics Job Simulation (Forage). Geldium is a fictional lender.*

**Purpose:** reproduce the key numbers quoted in the project reports: data-quality issues, how delinquency relates to each variable, segment rates, and a quick baseline model test.

**How to run:** save the dataset as `Delinquency_prediction_dataset.csv` in the same folder as this notebook (in Excel: File, Save As, CSV), then run all cells. The raw data is not included in this repository.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)

## 1. Load the data

In [2]:
try:
    df = pd.read_csv('Delinquency_prediction_dataset.csv')
except FileNotFoundError:
    df = pd.read_excel('Delinquency_prediction_dataset.xlsx')   # fallback if you kept the Excel file
print(df.shape)
df.head()

(500, 19)


,Customer_ID,Age,Income,Credit_Score,Credit_Utilization,Missed_Payments,Delinquent_Account,Loan_Balance,Debt_to_Income_Ratio,Employment_Status,Account_Tenure,Credit_Card_Type,Location,Month_1,Month_2,Month_3,Month_4,Month_5,Month_6
0,CUST0001,56,165580.0,398.0,0.390502,3,0,16310.0,0.317396,EMP,18,Student,Los Angeles,Late,Late,Missed,Late,Missed,Late
1,CUST0002,69,100999.0,493.0,0.312444,6,1,17401.0,0.196093,Self-employed,0,Standard,Phoenix,Missed,Missed,Late,Missed,On-time,On-time
2,CUST0003,46,188416.0,500.0,0.359930,0,0,13761.0,0.301655,Self-employed,1,Platinum,Chicago,Missed,Late,Late,On-time,Missed,Late
3,CUST0004,32,101672.0,413.0,0.371400,3,0,88778.0,0.264794,Unemployed,15,Platinum,Phoenix,Late,Missed,Late,Missed,Late,Late
4,CUST0005,60,38524.0,487.0,0.234716,2,0,13316.0,0.510583,Self-employed,11,Standard,Phoenix,Missed,On-time,Missed,Late,Late,Late


## 2. Data-quality checks
Before modelling, check what is missing, duplicated or inconsistent.

In [3]:
print('Missing values per column (only columns with missing data):')
missing = df.isna().sum()
print(missing[missing > 0], '\n')
print('Duplicate Customer_IDs:', df['Customer_ID'].duplicated().sum())
print('Credit_Score range:', df['Credit_Score'].min(), 'to', df['Credit_Score'].max())
print('Credit_Utilization above 100%:', (df['Credit_Utilization'] > 1).sum(), 'customers')
print('\nEmployment_Status labels as recorded:')
print(df['Employment_Status'].value_counts(dropna=False))

Missing values per column (only columns with missing data):
Income          39
Credit_Score     2
Loan_Balance    29
dtype: int64 

Duplicate Customer_IDs: 0
Credit_Score range: 301.0 to 847.0
Credit_Utilization above 100%: 4 customers

Employment_Status labels as recorded:
Employment_Status
Unemployed       93
retired          87
Employed         82
EMP              81
Self-employed    80
employed         77
Name: count, dtype: int64


Employment labels are inconsistent (for example `EMP`, `Employed`, `employed`). Standardise them, and build a few helper columns for later. The original data is not overwritten.

In [4]:
data = df.copy()
data['Employment_Status'] = data['Employment_Status'].str.lower().replace({'emp': 'employed'})
print(data['Employment_Status'].value_counts(), '\n')

# Payment history: On-time = 0, Late = 1, Missed = 2
month_cols = [f'Month_{i}' for i in range(1, 7)]
status_map = {'On-time': 0, 'Late': 1, 'Missed': 2}
for c in month_cols:
    data[c] = data[c].map(status_map)
data['late_count'] = (data[month_cols] >= 1).sum(axis=1)   # months that were late or missed

Employment_Status
employed         240
unemployed        93
retired           87
self-employed     80
Name: count, dtype: int64 



## 3. The target: how common is delinquency?

In [5]:
y = data['Delinquent_Account']
print(y.value_counts())
print(f'Delinquency rate: {y.mean():.1%}  ({y.sum()} of {len(y)} customers)')

Delinquent_Account
0    420
1     80
Name: count, dtype: int64
Delinquency rate: 16.0%  (80 of 500 customers)


## 4. Which variables relate to delinquency?
Correlation runs from -1 to +1; values near 0 mean almost no linear relationship.

In [6]:
numeric = ['Age', 'Income', 'Credit_Score', 'Credit_Utilization', 'Missed_Payments',
           'Loan_Balance', 'Debt_to_Income_Ratio', 'Account_Tenure', 'late_count'] + month_cols
corr = data[numeric + ['Delinquent_Account']].corr()['Delinquent_Account'].drop('Delinquent_Account')
print(corr.sort_values().round(3))
print('\nLargest absolute correlation:', round(corr.abs().max(), 3))

Account_Tenure         -0.040
Month_1                -0.039
Month_5                -0.033
Missed_Payments        -0.026
late_count             -0.024
Month_2                -0.021
Month_6                -0.015
Loan_Balance           -0.004
Month_3                 0.014
Age                     0.023
Credit_Utilization      0.034
Debt_to_Income_Ratio    0.034
Credit_Score            0.035
Income                  0.045
Month_4                 0.060
Name: Delinquent_Account, dtype: float64

Largest absolute correlation: 0.06


In [7]:
print('Average values: delinquent (1) vs not delinquent (0)')
data.groupby('Delinquent_Account')[['Credit_Score', 'Missed_Payments', 'Credit_Utilization',
                                     'Debt_to_Income_Ratio', 'Income']].mean().round(3)

Average values: delinquent (1) vs not delinquent (0)


,Credit_Score,Missed_Payments,Credit_Utilization,Debt_to_Income_Ratio,Income
Delinquent_Account,,,,,
0,575.146,2.99,0.489,0.297,107306.943
1,591.150,2.85,0.507,0.306,113902.013


Note: delinquent customers have *slightly fewer* missed payments on average, which is the opposite of what you would expect. That is a reason to check how the target is defined before trusting these fields.

## 5. Segment delinquency rates
Rate = share of customers in the segment who are delinquent. The chi-square p-value tests whether differences between segments could just be chance (p below 0.05 would suggest a real difference).

In [8]:
def segment_table(col):
    t = data.groupby(col)['Delinquent_Account'].agg(rate='mean', delinquent='sum', customers='count').round(3)
    p = chi2_contingency(pd.crosstab(data[col], y))[1]
    print(f'--- {col}   (chi-square p = {p:.3f}) ---')
    print(t, '\n')

data['Utilization_band'] = pd.cut(data['Credit_Utilization'], [0, .3, .5, .7, 2],
                                  labels=['up to 30%', '30-50%', '50-70%', 'over 70%'])
for col in ['Utilization_band', 'Credit_Card_Type', 'Employment_Status', 'Location']:
    segment_table(col)

--- Utilization_band   (chi-square p = 0.753) ---
                   rate  delinquent  customers
Utilization_band                              
up to 30%         0.120          10         83
30-50%            0.170          31        182
50-70%            0.169          28        166
over 70%          0.159          11         69 

--- Credit_Card_Type   (chi-square p = 0.354) ---
                   rate  delinquent  customers
Credit_Card_Type                              
Business          0.213          23        108
Gold              0.144          17        118
Platinum          0.118           9         76
Standard          0.128          11         86
Student           0.179          20        112 

--- Employment_Status   (chi-square p = 0.550) ---
                    rate  delinquent  customers
Employment_Status                              
employed           0.162          39        240
retired            0.115          10         87
self-employed      0.162          13      

In [9]:
# The comparisons quoted in the report
u = data['Credit_Utilization'] >= 0.30
print(f"Utilization 30%+ : {y[u].mean():.1%}  vs below 30%: {y[~u].mean():.1%}")
b = data['Credit_Card_Type'] == 'Business'
print(f"Business cards   : {y[b].mean():.1%}  vs other cards: {y[~b].mean():.1%}")
e = data['Employment_Status'] == 'unemployed'
print(f"Unemployed       : {y[e].mean():.1%}  vs all others: {y[~e].mean():.1%}")

Utilization 30%+ : 16.8%  vs below 30%: 12.0%
Business cards   : 21.3%  vs other cards: 14.5%
Unemployed       : 19.4%  vs all others: 15.2%


## 6. Baseline model test
A simple test of whether these variables can predict delinquency at all. **5-fold cross-validation** repeatedly trains on 80% of customers and tests on the other 20%, so every score comes from customers the model has not seen. **AUC** measures how well the model ranks delinquent customers above others: 0.5 is a coin flip, 1.0 is perfect.

In [10]:
num_features = ['Age', 'Income', 'Credit_Score', 'Credit_Utilization', 'Missed_Payments', 'Loan_Balance',
                'Debt_to_Income_Ratio', 'Account_Tenure', 'late_count'] + month_cols
cat_features = ['Employment_Status', 'Credit_Card_Type', 'Location']
X = data[num_features + cat_features]

prep = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Logistic regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random forest':       RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced', random_state=1),
    'Gradient boosting':   GradientBoostingClassifier(random_state=1)}

for name, model in models.items():
    scores = cross_val_score(Pipeline([('prep', prep), ('model', model)]), X, y, cv=cv, scoring='roc_auc')
    print(f'{name:20s} AUC = {scores.mean():.3f}  (+/- {scores.std():.3f} across folds)')

Logistic regression  AUC = 0.453  (+/- 0.077 across folds)


Random forest        AUC = 0.414  (+/- 0.075 across folds)


Gradient boosting    AUC = 0.399  (+/- 0.079 across folds)


## 7. What this shows, and what it does not

**Findings**
- All numeric correlations with delinquency are very small, and all three models score around 0.4 to 0.45 AUC. With only about 80 delinquent customers, the fold-to-fold spread (about 0.08) is large enough that these scores are **indistinguishable from chance (0.5)**. In plain terms: on this data, the variables do not predict delinquency.
- The segment gaps (utilization, business cards, unemployed) point in plausible directions, but none passes a significance test, so they are leads to test, not conclusions.
- Several data-quality issues need fixing or explaining before any real use.

**Limitations**
- Small sample (500 customers, about 80 delinquent); no model tuning; one train/test scheme.
- The data may be synthetic or may be missing the information that drives delinquency.
- This is a baseline check, not a production model.

**Recommended next step:** validate data definitions and test early-support outreach in a controlled pilot before relying on any model, as set out in the project reports.